#### PIP

In [165]:
%pip install -q nltk
%pip install -q spacy
%pip install -q gensim


[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Imports and Downloads

In [1]:
import nltk

nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('universal_tagset', quiet=True)

True

In [76]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import spacy
from spacy.tokens import Doc

# Load the spaCy model globally (en_core_web_sm is lightweight and efficient)
nlp = spacy.load("en_core_web_sm")

In [127]:
import numpy as np
import gensim.downloader as api

print("Global load: Downloading/Loading Word2Vec model...")
GLOBAL_W2V_MODEL = api.load('word2vec-google-news-300')
print("Global load: Word2Vec model ready.")

Global load: Downloading/Loading Word2Vec model...
Global load: Word2Vec model ready.


---

#### Set Notebook Seed

In [105]:
SEED=142

In [ ]:
import torch
import random
import numpy as np

def set_seed(seed: int = 142):
    """Locks all random number generators for exact reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed}")

In [111]:
set_seed(SEED)

Global seed set to 142


## Task 1

#### PropagandaFeaturePipeline (Class)

In [166]:
import re
import csv
from collections import Counter
import torch
import spacy
from spacy.tokens import Doc
from nltk.tag.perceptron import PerceptronTagger
from nltk.tag import map_tag

class PropagandaFeaturePipeline:
    """
    Encapsulates state (vocabularies, tagsets) while maintaining a pure 
    functional approach to row-by-row string processing and vectorization.
    """
    def __init__(self, spacy_model="en_core_web_sm", exclude_non_propaganda=True):

        self.LABELS = [
            'name_calling,labeling', 'repetition', 'causal_oversimplification', 
            'doubt', 'loaded_language', 'appeal_to_fear_prejudice', 
            'flag_waving', 'exaggeration,minimisation', 'not_propaganda'
        ]

        # For Task 1
        if exclude_non_propaganda:
            self.LABELS.remove('not_propaganda')
        
        self.UNIVERSAL_TAGSET = ["ADJ","ADP","ADV","CONJ","DET","NOUN","NUM","PRT","PRON","VERB",".","X"]
        
        # Custom (Shortened) NER Tagset
        self.NER_TAG = ['PERSON','ORG','GPE','DATE','NORP','CARDINAL','ORDINAL','TIME','LOC', 'O']
        
        # Top N most frequent words
        self.CUSTOM_STOPWORDS = ["the" , ",", "to", "of", "and", "in", "a", "that"]

        self.word_to_index = {}     # bow vector indicies
        self.word_to_index_silver = {}  # bow vector indicies using synthetic data
        self.hapax_words_list = []
        self.hapax_words_list_silver = []
        
        self.pos_to_index = self._build_tag_index(self.UNIVERSAL_TAGSET)    # POS tagset vector indicies
        self.ner_to_index = self._build_tag_index(["MISC"] + self.NER_TAG)  # NER tagset vector indicies

        self.nlp = spacy.load(spacy_model)  # taggers
        self.tagger = PerceptronTagger()

        self.w2v_model = None

    # ==========================================
    # INTERNAL BUILDER METHODS
    # ==========================================

    def _build_tag_index(self, tagset: list[str]) -> dict[str, int]:
        """Creates a mapping of tags to index positions."""
        return {tag: i for i, tag in enumerate(tagset) if tag not in ["O", "__BOUNDARY__"]}

    
    def build_vocabularies(self, gold_path: str, silver_path: str, full_context: str = True):
        """
        Parses the datasets to populate the class-level vocabulary matrices.
        This replaces the global counter loops from the notebook.
        """
        global_vocab = Counter()
        
        # Build Gold Vocab
        with open(gold_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in tsv_reader:
                label, text = self.process_row(row)
                if label not in self.LABELS: continue   # skip `not_propaganda` 
                tokens = self.tokenize_whole_words(text)

        

                if full_context:
                    global_vocab.update(tokens)
                else:    
                    in_snippet = False
                    for token in tokens:
                        if token == "<BOS>": in_snippet = True; continue
                        if token == "<EOS>": in_snippet = False; continue
                        if in_snippet:
                            global_vocab[token] += 1

        global_vocab_silver = global_vocab.copy()
                
        # Build Silver Vocab
        with open(silver_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in tsv_reader:
                label, text = self.process_row(row)
                if label not in self.LABELS: continue
                tokens = self.tokenize_whole_words(text) 
                
                in_snippet = False  # only draw silver counts from synthetic snippet
                for token in tokens:
                    if token == "<BOS>": in_snippet = True; continue
                    if token == "<EOS>": in_snippet = False; continue
                    if in_snippet and token in global_vocab:
                        global_vocab_silver[token] += 1

        # States
        self.hapax_words_list = [word for word, count in global_vocab.items() if count == 1]
        self.hapax_words_list_silver = [word for word, count in global_vocab_silver.items() if count == 1]
        
        gold_list = ["__UNK__"] + [word for word, count in global_vocab.items() if count > 1]
        silver_list = ["__UNK__"] + [word for word, count in global_vocab_silver.items() if count > 1]
        
        self.word_to_index = {w: i for i, w in enumerate(w for w in gold_list if w not in self.CUSTOM_STOPWORDS + ["<EOS>","<BOS>"])}
        self.word_to_index_silver = {w: i for i, w in enumerate(w for w in silver_list if w not in self.CUSTOM_STOPWORDS + ["<EOS>","<BOS>"])}
        print("Vocabulary State Successfully Initialized.")


    # ==========================================
    # FUNCTIONAL TEXT PROCESSING ZONE
    # ==========================================
    
    def universal_cleaning(self, raw_text: str) -> str:
        """Cleaning directly on raw STRING format"""
        text = raw_text.strip() # clear leading/trailing whitespace
        text = text.replace("\\'", "'").replace('\\"', '"') # strip out python escape backslashes
        text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'") # standardize quotes to flat quotes
        text = re.sub(r"(?<=\w)'(?=\w)|(?<=[sS])'", '', text) # collapse intra-word apostrophes: won't -> wont, lukes' -> lukes
        text = re.sub(r'[\\/\[\]*|@\ \-.:$#+=]', ' ', text) # remove artifacts: \ / [ ] * | @ space - . : $ # + =
        text = text.replace("<BOS>", " <BOS> ").replace("<EOS>", " <EOS> ") # ensure space around bound tags
        return " ".join(text.split())

    def process_row(self, row: dict) -> tuple[str, str]:
        """Process raw row directly from csv"""
        return row['label'], self.universal_cleaning(row['tagged_in_context'])

    def tokenize_whole_words(self, text: str) -> list[str]:
        """Turn string text into whole-word tokens using regex parser"""
        localized_text = re.sub(r'\b\d+(?:,\d+)*\b', 'num', text)
        pattern = r"<BOS>|<EOS>|(?:[a-zA-Z]\.)+|[a-zA-Z0-9]+(?:[-']?[a-zA-Z0-9]+)*|[^\w\s]"
        raw_tokens = re.findall(pattern, localized_text)
        return [t if t in ["<BOS>", "<EOS>"] else t.lower() for t in raw_tokens]

    def tag_pos_pipeline(self, text: str) -> list[str]:
        """Turn string text into pos tokens using NLTK Perceptron"""
        tokens = self.tokenize_whole_words(text)
        raw_tags = self.tagger.tag(tokens) # nltk PerceptronTagger
        return [
            ("__BOUNDARY__") if t == "<BOS>" or t == "<EOS>" else
            ("NUM") if t.lower() == "num" else # capture num rule from string formatting
            (".") if t in ['"', "'", '`'] else # override mapping
            (map_tag('en-ptb', 'universal', tag)) # map from perceptron native pentree to universal tags
            for t, tag in raw_tags
        ]

    def tag_ner_pipeline(self, text: str) -> list[str]:
        """Turn string text into NER tokens using Spacy"""
        
        allowed = set(self.NER_TAG)
        tokens = self.tokenize_whole_words(text)

        doc = Doc(self.nlp.vocab, words=tokens)
        for name, proc in self.nlp.pipeline: doc = proc(doc)
            
        ner_tags = []
        for token in doc:
            if token.text in ["<BOS>", "<EOS>"]: ner_tags.append("__BOUNDARY__")
            elif token.ent_type_:
                # Custom NER list excludes low count tags, route these into MISC category
                ner_tags.append(f"{token.ent_type_}" if token.ent_type_ in allowed else "MISC")
            else: ner_tags.append("O")
        return ner_tags


    # ==========================================
    # VECTORIZATION ZONE
    # ==========================================

    def string_to_word2vec_vector(self, string: str, use_silver: bool = False) -> list[float]:
        """
        Turn text string into a 300D Word2Vec mean-pooled vector.
        Only processes words that exist in our learned vocabularies.
        """
        if self.w2v_model is None:
            self.load_word2vec()

        active_vocab = self.word_to_index_silver if use_silver else self.word_to_index
        tokenized = self.tokenize_whole_words(string)
        
        vectors = []
        unk_count = 0
        for token in tokenized:
            if token in ["<EOS>", "<BOS>"] or token in self.CUSTOM_STOPWORDS: continue  
            if token not in active_vocab: 
                unk_count += 1 
                continue 
                
            if token in self.w2v_model:
                vectors.append(self.w2v_model[token])
            elif token.capitalize() in self.w2v_model: # fallback check cap version
                vectors.append(self.w2v_model[token.capitalize()])
                # will not match names, punct or obsurce words
                
        if len(vectors) > 0:
            mean_vector = np.mean(vectors, axis=0).tolist() # Mean Pooling
        else:
            mean_vector = [0.0] * 300 # no match fallback
            print("empty w2v vector")
        
        unk_count = unk_count / len(tokenized) if len(tokenized) > 0 else 0.0 # normalize
            
        return mean_vector, unk_count


    def string_to_bow_vector(self, string: str, use_silver: bool = False) -> list[int]:
        """Turn text string into a vocab bow python vector"""
        active_vocab = self.word_to_index_silver if use_silver else self.word_to_index
        tokenized = self.tokenize_whole_words(string)
        sequence_vector = [0] * len(active_vocab)

        for token in tokenized:
            # avoiding counting boundaries and stopwords
            if token in ["<EOS>", "<BOS>"] or token in self.CUSTOM_STOPWORDS: continue
            
            # populate sparse vector + unk index
            idx = active_vocab.get(token, active_vocab["__UNK__"])
            sequence_vector[idx] += 1

        return sequence_vector


    def tagset_to_vector(self, string: str, tag_type: str) -> list[int]:
        """Turn text string into a tagset bow python vector"""
        if tag_type == "POS":
            tag_list = self.tag_pos_pipeline(string) 
            active_index = self.pos_to_index
        elif tag_type == "NER":
            tag_list = self.tag_ner_pipeline(string)
            active_index = self.ner_to_index

        sequence_vector = [0] * len(active_index)
        for tag in tag_list:
            if tag in ["__BOUNDARY__", "O"]: continue
            sequence_vector[active_index[tag]] += 1
        return sequence_vector


    def string_to_input_vector(
            self, 
            string_text: str, 
            use_silver: bool = False,
            feature_type: str = "bow"
            ) -> torch.Tensor:
        """
        Routes text string to the correct token vectorization method,
        generates POS/NER vectors, and concatenates them for MLP input.
        """

        # Generate token vectors
        if feature_type == "word2vec":
            text_vector, unk_count = self.string_to_word2vec_vector(string_text, use_silver)
            t_vocab = torch.tensor(text_vector + [unk_count], dtype=torch.float32)
            
        elif feature_type == "bow":
            text_vector = self.string_to_bow_vector(string_text, use_silver)
            t_vocab = torch.tensor(text_vector, dtype=torch.float32)

        # Generate tag vectors
        pos_vector = self.tagset_to_vector(string_text, "POS")
        ner_vector = self.tagset_to_vector(string_text, "NER")

        # pytorch tensor normalisation
        t_pos = torch.tensor(pos_vector, dtype=torch.float32)
        t_ner = torch.tensor(ner_vector, dtype=torch.float32)

        if feature_type == "word2vec":
            # convert tagset counts to distrubution to match w2v magnitutde
            if t_pos.sum() > 0: t_pos = t_pos / t_pos.sum()
            if t_ner.sum() > 0: t_ner = t_ner / t_ner.sum() # converts space into distribtuion

        x_combined = torch.cat([t_vocab, t_pos, t_ner], dim=0)

        return x_combined.unsqueeze(0)
    

    # ==========================================
    # LOADING ZONE
    # ==========================================

    def load_word2vec(self):
        """Pulls the pre-loaded global Word2Vec model."""
        if self.w2v_model is None:
            global GLOBAL_W2V_MODEL
            if 'GLOBAL_W2V_MODEL' in globals() and GLOBAL_W2V_MODEL is not None:
                self.w2v_model = GLOBAL_W2V_MODEL
                print("Word2Vec model successfully linked from global scope.")
            else:
                raise RuntimeError(
                    "GLOBAL_W2V_MODEL is not defined. Please run the global loading cell at the top of the notebook first."
                )

---

##### Example Run:

In [113]:
set_seed(SEED)

pipeline = PropagandaFeaturePipeline()

pipeline.build_vocabularies(
    gold_path='../data/propaganda_train_100.tsv', 
    silver_path='../data/silver_train.tsv', 
    full_context=True
)

print(f"List of corpus labels:             {pipeline.LABELS}")
print(f"Universal POS Tagset:              {pipeline.UNIVERSAL_TAGSET}")
print(f"Custom Simplified NER tagset:      {pipeline.NER_TAG}")
print(f"Custom Stopword List:              {pipeline.CUSTOM_STOPWORDS}")
print(f"Dims of baseline gold sparse vec:  {len(pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(pipeline.ner_to_index)}")

print(f"POS Tagger:                        {pipeline.tagger}")
print(f"NER Tagger:                        {pipeline.nlp}")


Global seed set to 142
Vocabulary State Successfully Initialized.
List of corpus labels:             ['name_calling,labeling', 'repetition', 'causal_oversimplification', 'doubt', 'loaded_language', 'appeal_to_fear_prejudice', 'flag_waving', 'exaggeration,minimisation']
Universal POS Tagset:              ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRT', 'PRON', 'VERB', '.', 'X']
Custom Simplified NER tagset:      ['PERSON', 'ORG', 'GPE', 'DATE', 'NORP', 'CARDINAL', 'ORDINAL', 'TIME', 'LOC', 'O']
Custom Stopword List:              ['the', ',', 'to', 'of', 'and', 'in', 'a', 'that']
Dims of baseline gold sparse vec:  377
Dims of gold + silver sparse vec:  869
Gold Only Singletons:              858
Silver Enriched Singletons:        366
Dims of POS vector:                12
Dimes of NER vector:               10
POS Tagger:                        <nltk.tag.perceptron.PerceptronTagger object at 0x15ab115d0>
NER Tagger:                        <spacy.lang.en.English object at 0x41ea5ae

#### PropagandaTrainer (Class)

In [167]:
import csv
import torch
import torch.nn as nn
import random

class PropagandaTrainer:
    """
    Builds the standardized PyTorch MLP architecture,
    configures optimization, 
    and executes the streaming training loop.
    """
    def __init__(
        self, 
        pipeline, 
        hidden_dim: int = 64, 
        dropout_p: float = 0.3,
        lr: float = 0.0005,
        weight_decay: float = 0.05,
        use_silver: bool = False,    # vocab selector
        feature_type: str = "bow"
    ):
        self.pipeline = pipeline
        self.use_silver = use_silver
        self.feature_type = feature_type
        self.label_to_idx = {label: i for i, label in enumerate(pipeline.LABELS)}
        self.best_val_loss = float('inf')
        
        # Dynamically compute total input dimension from pipeline state
        if feature_type == "word2vec":
            self.pipeline.load_word2vec()
            token_dim = 301
        else:
            active_vocab = pipeline.word_to_index_silver if use_silver else pipeline.word_to_index
            token_dim = len(active_vocab)
        
        # token + tagset vectors
        input_dimension = (
            token_dim + 
            len(pipeline.pos_to_index) + 
            len(pipeline.ner_to_index)
        )
        
        # Build classification head
        self.model = self._build_head(
            input_dim=input_dimension,
            hidden_dim=hidden_dim,
            num_classes=len(pipeline.LABELS),
            dropout_p=dropout_p
        )
        
        # 3. Configure head components
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(), 
            lr=lr, 
            weight_decay=weight_decay
        )

    # ==========================================
    # NETWORK BUILDER
    # ==========================================
    def _build_head(self, input_dim: int, hidden_dim: int, num_classes: int, dropout_p: float) -> nn.Module:
        """
        Constructs the standardized MLP classification head directly.
        Uses LayerNorm instead of BatchNorm1d to ensure stability during batch_size=1 streaming.
        """
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, num_classes)
        )

    # ==========================================
    # STREAMING TRAINING PASS
    # ==========================================
    def run_training_loop(
        self, 
        dataset_path: str, 
        epochs: int = 5, 
        save_path: str = 'propaganda_mlp_weights.pt',
        save_best_only: bool = True
    ) -> list[tuple[int, float, float]]:
        """
        Executes row-by-row streaming training and 10% modulo validation.
        """

        epoch_history = []

        for epoch in range(1, epochs + 1):
            print(f"--- Starting Epoch {epoch} ---")

            running_train_loss, train_samples = 0.0, 0
            running_val_loss, val_samples = 0.0, 0

            with open(dataset_path, mode='r', encoding='utf-8') as file:
                tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
                
                for row_idx, raw_row in enumerate(tsv_reader, start=1):
                    label, text = self.pipeline.process_row(raw_row)

                    # Formatting guardrail
                    if text.count("<BOS>") != 1 or text.count("<EOS>") != 1 or label not in self.label_to_idx:
                        continue

                    # Feature Extraction
                    x_batched = self.pipeline.string_to_input_vector(text, use_silver=self.use_silver, feature_type=self.feature_type)
                    y_target = torch.tensor([self.label_to_idx[label]], dtype=torch.long)
                    
                    # 10% Modulo Split for internal dev validation
                    if row_idx % 10 == 0:
                        self.model.eval() # testing
                        with torch.no_grad():
                            logits = self.model(x_batched)
                            val_loss = self.criterion(logits, y_target)
                            running_val_loss += val_loss.item()
                            val_samples += 1
                    else:
                        self.model.train() # training
                        self.optimizer.zero_grad()
                        logits = self.model(x_batched)
                        loss = self.criterion(logits, y_target)
                        loss.backward()
                        self.optimizer.step()
                        
                        running_train_loss += loss.item()
                        train_samples += 1

            # Epoch reporting
            epoch_train_loss = running_train_loss / train_samples if train_samples > 0 else 0.0
            epoch_val_loss = running_val_loss / val_samples if val_samples > 0 else 0.0
            
            print(f"Epoch {epoch} Results | Avg Train Loss: {epoch_train_loss:.4f} | Avg Dev Loss: {epoch_val_loss:.4f}")

            epoch_history.append((epoch, round(epoch_train_loss, 4), round(epoch_val_loss, 4)))

            if save_best_only:
                # Early stopping / Best Checkpoint Behavior
                if epoch_val_loss < self.best_val_loss:
                    self.best_val_loss = epoch_val_loss
                    torch.save(self.model.state_dict(), save_path)
                    print(f"--> New best validation loss ({self.best_val_loss:.4f}) achieved! Model saved to {save_path}.\n")
                else:
                    print(f"--> No improvement on validation loss. Skipping save.\n")
            else:
                # Fixed N-Epoch Behavior: Always overwrite state dict at every epoch
                torch.save(self.model.state_dict(), save_path)
                print(f"--> Model state updated at Epoch {epoch} and saved to {save_path}.\n")
            
        return epoch_history

    # ==========================================
    # MODEL EVALUATION / INFERENCE PASS
    # ==========================================
    def evaluate(
        self, 
        dataset_path: str, 
        weights_path: str = None, 
        random_guess: bool = False
    ) -> dict:
        """
        Loads saved weights from disk and streams a test/validation file
        to return predictions, targets, and classification metrics.
        """

        # model type router, inc baseline
        if random_guess:
            random.seed(100)
            mode_name = "RANDOM GUESSING BASELINE"
        else:
            if weights_path is None:
                raise ValueError("weights_path must be provided when random_guess=False")
            mode_name = f"MODEL EVALUATION ({weights_path})"
            self.model.load_state_dict(torch.load(weights_path, weights_only=True))
            self.model.eval()

        all_preds = []
        all_targets = []
        idx_to_label = {i: label for label, i in self.label_to_idx.items()}
        num_classes = len(self.label_to_idx)

        # Stream dataset and gather predictions
        with open(dataset_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            
            for row_idx, raw_row in enumerate(tsv_reader, start=1):
                label, text = self.pipeline.process_row(raw_row)

                if text.count("<BOS>") != 1 or text.count("<EOS>") != 1 or label not in self.label_to_idx:
                    continue
                
                y_target = self.label_to_idx[label]

                if random_guess:
                    predicted_idx = random.randint(0, num_classes - 1)
                else:
                    x_batched = self.pipeline.string_to_input_vector(
                        text, 
                        use_silver=self.use_silver,
                        feature_type=self.feature_type
                    )
                    with torch.no_grad():
                        logits = self.model(x_batched)
                        predicted_idx = torch.argmax(logits, dim=1).item()
                
                all_preds.append(predicted_idx)
                all_targets.append(y_target)
    
        # Calculate Performance Metrics
        from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

        # Overall Metrics
        acc = accuracy_score(all_targets, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_targets, all_preds, average='macro', zero_division=0
        )

        print("\n" + "="*50)
        print(f" EVALUATION REPORT: {weights_path}")
        print("="*50)
        print(f" Accuracy:  {acc:.4f}")
        print(f" Macro Precision: {precision:.4f}")
        print(f" Macro Recall:    {recall:.4f}")
        print(f" Macro F1 Score:  {f1:.4f}")
        print("="*50 + "\n")

        # Detailed per-class breakdown
        target_names = [idx_to_label[i] for i in sorted(idx_to_label.keys())]
        print(classification_report(all_targets, all_preds, target_names=target_names, zero_division=0))

        return {
            "accuracy": acc,
            "macro_f1": f1,
            "predictions": [idx_to_label[p] for p in all_preds],
            "targets": [idx_to_label[t] for t in all_targets]
        }

---

##### Example run:

In [115]:
set_seed(SEED)

# 1. Initialize and build feature pipeline state
test_pipeline = PropagandaFeaturePipeline()
test_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv'
)

# 2. Instantiate trainer
test_trainer = PropagandaTrainer(
    pipeline=test_pipeline,
    hidden_dim=64,
    dropout_p=0.3,
    lr=0.0005,
    weight_decay=0.05,
    use_silver=False,
    feature_type="bow"
)

# 3. Launch dynamic training pass
test_trainer.run_training_loop(
    dataset_path='../data/propaganda_train_100.tsv',
    epochs=3,
    save_path='test_propaganda_mlp_weights.pt'
)

Global seed set to 142
Vocabulary State Successfully Initialized.
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.2213 | Avg Dev Loss: 2.1621
--> New best validation loss (2.1621) achieved! Model saved to test_propaganda_mlp_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.6120 | Avg Dev Loss: 2.1611
--> New best validation loss (2.1611) achieved! Model saved to test_propaganda_mlp_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 0.9761 | Avg Dev Loss: 2.2974
--> No improvement on validation loss. Skipping save.



[(1, 2.2213, 2.1621), (2, 1.612, 2.1611), (3, 0.9761, 2.2974)]

---

### Evaluation

1. [Random Guessing Baseline]()
---
1. [BoW: Full Context, Gold Only](#bow-baseline-full-context-gold-only)
2. [BoW: Full Context, Silver Enriched]()
3. [BoW: Snippet, Gold Only]()
4. [BoW: Snippet, Silver Enriched]()
---
1. [W2V: Full Context, Gold Only]()
2. [W2V: Full Context, Silver Enriched]()
3. [W2V: Snippet, Gold Only]()
4. [W2V: Snippet, Silver Enriched]()
---

#### Random Guessing Baseline

In [119]:
set_seed(SEED)

# Instantiate trainer shell (reuses pipeline setup)
eval_trainer = PropagandaTrainer(pipeline=pipeline)

# 1. Random Guessing Baseline Evaluation
random_results = eval_trainer.evaluate(
    dataset_path='../data/propaganda_val.tsv',
    random_guess=True
)

Global seed set to 142

 EVALUATION REPORT: None
 Accuracy:  0.1392
 Macro Precision: 0.1404
 Macro Recall:    0.1388
 Macro F1 Score:  0.1385

                           precision    recall  f1-score   support

    name_calling,labeling       0.11      0.15      0.13        34
               repetition       0.14      0.15      0.15        40
causal_oversimplification       0.15      0.17      0.16        35
                    doubt       0.15      0.14      0.14        43
          loaded_language       0.14      0.13      0.14        39
 appeal_to_fear_prejudice       0.21      0.16      0.18        43
              flag_waving       0.14      0.11      0.12        45
exaggeration,minimisation       0.08      0.10      0.09        30

                 accuracy                           0.14       309
                macro avg       0.14      0.14      0.14       309
             weighted avg       0.14      0.14      0.14       309



#### BoW Experiments

##### HyperParameter Sweep: BoW Baseline

In [116]:
import itertools

set_seed(SEED)

pipeline = PropagandaFeaturePipeline()
pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=True
)

# 2. Parameter Search Ranges
param_grid = {
    'hidden_dim': [64, 128],
    'lr': [0.001, 0.0005, 0.0001],
    'dropout_p': [0.3, 0.5]
}

# 3. Generate Cartesian combinations
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Tracking metrics
global_best_loss = float('inf')
best_config = None
sweep_results = []

USE_SILVER = False

# 4. Execute Hyperparameter Sweep
for run_id, config in enumerate(experiments, start=1):
    print(f"\n==================================================")
    print(f" SWEEP RUN {run_id}/{len(experiments)} | {config}")
    print(f"==================================================")

    set_seed(SEED)
    
    trainer = PropagandaTrainer(
        pipeline=pipeline,
        hidden_dim=config["hidden_dim"],
        dropout_p=config["dropout_p"],
        lr=config["lr"],
        use_silver=USE_SILVER    # vocab
    )

    save_filename = f"./param_sweep/sweep_gold_run_{run_id}.pt"

    epoch_tuples = trainer.run_training_loop(
        dataset_path='../data/propaganda_train.tsv',
        epochs=5,
        save_path=save_filename,
        save_best_only=True
    )

    # Capture overall top-performing configuration
    if trainer.best_val_loss < global_best_loss:
        global_best_loss = trainer.best_val_loss
        best_config = config
        print(f"🔥 NEW BEST MODEL FOUND! Dev Loss: {global_best_loss:.4f}")

    run_row = [
        run_id,
        config["hidden_dim"],
        config["lr"],
        config["dropout_p"],
        USE_SILVER,
        epoch_tuples  # Contains [(1, train_l, dev_l), (2, train_l, dev_l), ...]
    ]

    sweep_results.append(run_row)

print("\n" + "="*50)
print(f"SWEEP COMPLETE!")
print(f"Lowest Validation Loss: {global_best_loss:.4f}")
print(f"Optimal Hyperparameter Set: {best_config}")
print("="*50)

# ========================================================
# Save Sweep Results Directly to CSV File
# ========================================================
csv_filename = "./param_sweep/sweep_results_gold.csv"
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    
    # 1. Construct the header dynamically
    header = ["run_id", "hidden_dim", "lr", "dropout_p", "use_silver"]
    # Add columns for up to 5 epochs
    for i in range(1, 6):
        header.extend([f"train_loss_{i}", f"val_loss_{i}"])
    
    writer.writerow(header)
    
    # 2. Flatten the data for each run
    for row in sweep_results:
        run_id, hidden, lr, dropout, silver, history = row
        
        # Start the flat row with your metadata
        flat_row = [run_id, hidden, lr, dropout, silver]
        
        # Extract losses from each epoch tuple (epoch, train, val)
        for epoch_data in history:
            _, train_loss, val_loss = epoch_data
            flat_row.extend([train_loss, val_loss])
            
        writer.writerow(flat_row)

print(f"\nSweep complete, saved {len(sweep_results)} to '{csv_filename}'.")


Global seed set to 142
Vocabulary State Successfully Initialized.

 SWEEP RUN 1/12 | {'hidden_dim': 64, 'lr': 0.001, 'dropout_p': 0.3}
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.0650 | Avg Dev Loss: 2.0721
--> New best validation loss (2.0721) achieved! Model saved to ./param_sweep/sweep_gold_run_1.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.3971 | Avg Dev Loss: 1.9636
--> New best validation loss (1.9636) achieved! Model saved to ./param_sweep/sweep_gold_run_1.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 0.7827 | Avg Dev Loss: 2.1162
--> No improvement on validation loss. Skipping save.

--- Starting Epoch 4 ---
Epoch 4 Results | Avg Train Loss: 0.4696 | Avg Dev Loss: 2.4174
--> No improvement on validation loss. Skipping save.

--- Starting Epoch 5 ---
Epoch 5 Results | Avg Train Loss: 0.3612 | Avg Dev Loss: 2.5464
--> No improvement on validation loss. Skipping save.

🔥 NEW BEST MODEL FOUND! Dev Loss: 

In [117]:
HIDDEN_DIMS = 128
LR = 0.0001
DROPOUT = 0.5
EPOCHS = 3

##### BoW Baseline: Full Context, Gold Only

In [ ]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = False

MODEL_SAVE = 'bow_full_gold'

bow_full_gold_pipeline = PropagandaFeaturePipeline()
bow_full_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_full_gold_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_full_gold_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_full_gold_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_full_gold_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_full_gold_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_full_gold_pipeline.ner_to_index)}")

bow_full_gold_eval_trainer = PropagandaTrainer(
    pipeline=bow_full_gold_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_full_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)

print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_full_gold_val_results = bow_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_full_gold_train_results = bow_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  3265
Dims of gold + silver sparse vec:  4002
Gold Only Singletons:              3038
Silver Enriched Singletons:        2301
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1974 | Avg Dev Loss: 2.0956
--> Model state updated at Epoch 1 and saved to ./final_models/bow_full_gold_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.7997 | Avg Dev Loss: 1.9858
--> Model state updated at Epoch 2 and saved to ./final_models/bow_full_gold_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.3216 | Avg Dev Loss: 1.9269
--> Model state updated at Epoch 3 and saved to ./final_models/bow_full_gold_weights.pt.

The results of the bow_full_gold model on the validation set:

 EVALUATION REPORT: ./final_models/bow_full_gold_weights.pt
 Accuracy:  0.3

##### BoW Baseline: Snippet, Gold Only

In [ ]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False     # Snippet-only
USE_SILVER = False

MODEL_SAVE = 'bow_snippet_gold'

bow_snippet_gold_pipeline = PropagandaFeaturePipeline()

bow_snippet_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_snippet_gold_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_snippet_gold_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_snippet_gold_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_snippet_gold_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_snippet_gold_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_snippet_gold_pipeline.ner_to_index)}")

bow_snippet_gold_eval_trainer = PropagandaTrainer(
    pipeline=bow_snippet_gold_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_snippet_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)

print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_snippet_gold_val_results = bow_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_snippet_gold_train_results = bow_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  1483
Dims of gold + silver sparse vec:  2415
Gold Only Singletons:              2051
Silver Enriched Singletons:        1119
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1714 | Avg Dev Loss: 2.0250
--> Model state updated at Epoch 1 and saved to ./final_models/bow_snippet_gold_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.8766 | Avg Dev Loss: 1.9603
--> Model state updated at Epoch 2 and saved to ./final_models/bow_snippet_gold_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.6214 | Avg Dev Loss: 1.8933
--> Model state updated at Epoch 3 and saved to ./final_models/bow_snippet_gold_weights.pt.

The results of the bow_snippet_gold model on the validation set:

 EVALUATION REPORT: ./final_models/bow_snippet_gold_weights.pt


##### BoW Baseline: Full Context, Silver Enriched

In [ ]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = True

MODEL_SAVE = 'bow_full_silver'

bow_full_silver_pipeline = PropagandaFeaturePipeline()

bow_full_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_full_silver_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_full_silver_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_full_silver_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_full_silver_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_full_silver_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_full_silver_pipeline.ner_to_index)}")

bow_full_silver_eval_trainer = PropagandaTrainer(
    pipeline=bow_full_silver_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_full_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)


print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_full_silver_val_results = bow_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_full_silver_train_results = bow_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  3265
Dims of gold + silver sparse vec:  4002
Gold Only Singletons:              3038
Silver Enriched Singletons:        2301
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1956 | Avg Dev Loss: 2.1225
--> Model state updated at Epoch 1 and saved to ./final_models/bow_full_silver_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.7399 | Avg Dev Loss: 2.0029
--> Model state updated at Epoch 2 and saved to ./final_models/bow_full_silver_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.1995 | Avg Dev Loss: 1.9584
--> Model state updated at Epoch 3 and saved to ./final_models/bow_full_silver_weights.pt.

The results of the bow_full_silver model on the validation set:

 EVALUATION REPORT: ./final_models/bow_full_silver_weights.pt
 Accu

##### BoW Baseline: Snippet, Silver Enriched

In [ ]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False
USE_SILVER = True

MODEL_SAVE = 'bow_snippet_silver'

bow_snippet_silver_pipeline = PropagandaFeaturePipeline()

bow_snippet_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_snippet_silver_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_snippet_silver_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_snippet_silver_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_snippet_silver_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_snippet_silver_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_snippet_silver_pipeline.ner_to_index)}")

bow_snippet_silver_eval_trainer = PropagandaTrainer(
    pipeline=bow_snippet_silver_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_snippet_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)

print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_snippet_silver_val_results = bow_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_snippet_silver_train_results = bow_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  1483
Dims of gold + silver sparse vec:  2415
Gold Only Singletons:              2051
Silver Enriched Singletons:        1119
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1746 | Avg Dev Loss: 2.0444
--> Model state updated at Epoch 1 and saved to ./final_models/bow_snippet_silver_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.7953 | Avg Dev Loss: 1.9664
--> Model state updated at Epoch 2 and saved to ./final_models/bow_snippet_silver_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.4224 | Avg Dev Loss: 1.9426
--> Model state updated at Epoch 3 and saved to ./final_models/bow_snippet_silver_weights.pt.

The results of the bow_snippet_silver model on the validation set:

 EVALUATION REPORT: ./final_models/bow_snippet_silver_w

#### Word2Vec Experiments

##### Hyperparameter Sweep: Word2Vec

In [169]:
import itertools
import csv

# 1. Pipeline Initialization & Vocab Construction
set_seed(SEED)

w2v_sweep_pipeline = PropagandaFeaturePipeline()
w2v_sweep_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=True
)

# 2. Search Grid Specifically Tailored for Dense Word2Vec Embeddings
param_grid = {
    'hidden_dim': [64, 128],
    'lr': [0.005, 0.001, 0.0005],
    'dropout_p': [0.3, 0.5]
}

# 3. Cartesian Product Combinations
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Track Metrics
global_best_loss = float('inf')
best_config = None
w2v_sweep_results = []

USE_SILVER = False
FEATURE_TYPE = "word2vec"
EPOCHS = 5

# 4. Execute Sweep
for run_id, config in enumerate(experiments, start=1):
    print(f"\n==================================================")
    print(f" WORD2VEC SWEEP RUN {run_id}/{len(experiments)} | {config}")
    print(f"==================================================")

    # Lock seed per run so every network configuration starts with identical weight initialization
    set_seed(SEED)
    
    trainer = PropagandaTrainer(
        pipeline=w2v_sweep_pipeline,
        hidden_dim=config["hidden_dim"],
        dropout_p=config["dropout_p"],
        lr=config["lr"],
        use_silver=USE_SILVER,
        feature_type=FEATURE_TYPE
    )

    save_filename = f"./param_sweep/sweep_w2v_gold_run_{run_id}.pt"

    epoch_tuples = trainer.run_training_loop(
        dataset_path='../data/propaganda_train.tsv',
        epochs=EPOCHS,
        save_path=save_filename,
        save_best_only=True
    )

    # Capture overall top-performing configuration
    if trainer.best_val_loss < global_best_loss:
        global_best_loss = trainer.best_val_loss
        best_config = config
        print(f"🔥 NEW BEST WORD2VEC MODEL FOUND! Dev Loss: {global_best_loss:.4f}")

    run_row = [
        run_id,
        config["hidden_dim"],
        config["lr"],
        config["dropout_p"],
        USE_SILVER,
        epoch_tuples  # [(epoch, train_loss, dev_loss), ...]
    ]

    w2v_sweep_results.append(run_row)

print("\n" + "="*50)
print(f"WORD2VEC SWEEP COMPLETE!")
print(f"Lowest Validation Loss: {global_best_loss:.4f}")
print(f"Optimal Hyperparameter Set: {best_config}")
print("="*50)

Global seed set to 142
Vocabulary State Successfully Initialized.

 WORD2VEC SWEEP RUN 1/12 | {'hidden_dim': 64, 'lr': 0.005, 'dropout_p': 0.3}
Global seed set to 142
Word2Vec model successfully linked from global scope.
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1256 | Avg Dev Loss: 2.1069
--> New best validation loss (2.1069) achieved! Model saved to ./param_sweep/sweep_w2v_gold_run_1.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 2.0179 | Avg Dev Loss: 2.0426
--> New best validation loss (2.0426) achieved! Model saved to ./param_sweep/sweep_w2v_gold_run_1.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.9436 | Avg Dev Loss: 2.0212
--> New best validation loss (2.0212) achieved! Model saved to ./param_sweep/sweep_w2v_gold_run_1.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.8882 |

In [174]:
W2W_HIDDEN_DIMS = 64
W2W_LR = 0.0005
W2W_DROPOUT = 0.5
W2W_EPOCHS = 5

##### Word2Vec: Full Context, Gold Only

In [175]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = False
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_full_gold'

# 1. Initialize Pipeline & Build Vocabularies
w2v_full_gold_pipeline = PropagandaFeaturePipeline()
w2v_full_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_full_gold_eval_trainer = PropagandaTrainer(
    pipeline=w2v_full_gold_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_full_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_full_gold_val_results = w2v_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_full_gold_train_results = w2v_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1292 | Avg Dev Loss: 2.0802
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_full_gold_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9767 | Avg Dev Loss: 2.0237
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_full_gold_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8775 | Avg Dev Loss: 1.9840
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_full_gold_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7903 | Avg Dev Loss: 1.9405
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_full_gold_weights.pt.

--- S

##### Word2Vec: Snippet, Gold Only

In [176]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False
USE_SILVER = False
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_snippet_gold'

# 1. Initialize Pipeline & Build Vocabularies
w2v_snippet_gold_pipeline = PropagandaFeaturePipeline()
w2v_snippet_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_snippet_gold_eval_trainer = PropagandaTrainer(
    pipeline=w2v_snippet_gold_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_snippet_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_snippet_gold_val_results = w2v_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_snippet_gold_train_results = w2v_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1304 | Avg Dev Loss: 2.0899
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_snippet_gold_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9802 | Avg Dev Loss: 2.0189
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_snippet_gold_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8753 | Avg Dev Loss: 1.9670
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_snippet_gold_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7833 | Avg Dev Loss: 1.9213
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_snippet_gold_weight

##### Word2Vec: Full Context, Silver Enriched

In [177]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = True
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_full_silver'

# 1. Initialize Pipeline & Build Vocabularies
w2v_full_silver_pipeline = PropagandaFeaturePipeline()
w2v_full_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_full_silver_eval_trainer = PropagandaTrainer(
    pipeline=w2v_full_silver_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_full_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_full_silver_val_results = w2v_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_full_silver_train_results = w2v_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1302 | Avg Dev Loss: 2.0805
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_full_silver_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9815 | Avg Dev Loss: 2.0229
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_full_silver_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8802 | Avg Dev Loss: 1.9751
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_full_silver_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7937 | Avg Dev Loss: 1.9083
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_full_silver_weights.pt

##### Word2Vec: Snippet, Silver Enriched

In [178]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False
USE_SILVER = True
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_snippet_silver'

# 1. Initialize Pipeline & Build Vocabularies
w2v_snippet_silver_pipeline = PropagandaFeaturePipeline()
w2v_snippet_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_snippet_silver_eval_trainer = PropagandaTrainer(
    pipeline=w2v_snippet_silver_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_snippet_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_snippet_silver_val_results = w2v_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_snippet_silver_train_results = w2v_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1255 | Avg Dev Loss: 2.0810
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_snippet_silver_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9715 | Avg Dev Loss: 1.9991
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_snippet_silver_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8633 | Avg Dev Loss: 1.9484
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_snippet_silver_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7724 | Avg Dev Loss: 1.8982
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_snippet_silve

## Task 2

In [ ]:
# ==========================================
# CELL 1: Installations
# ==========================================
%pip install -q transformers sentencepiece
%pip install -q pytorch-crf
%pip install -q scikit-learn

In [ ]:
# ==========================================
# CELL 2: Imports & Global Configuration
# ==========================================
import re
import csv
import torch
import random
import string
import numpy as np
import torch.nn as nn
from transformers import AutoTokenizer, DebertaV2Model
from torchcrf import CRF
from sklearn.metrics import precision_recall_fscore_support, classification_report

# Inherited Seed Logic for Reproducibility
def set_seed(seed: int = 142):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

SEED = 142
set_seed(SEED)

# Task 2 Label Schemas
TECHNIQUES = [
    'flag_waving', 'appeal_to_fear_prejudice', 'causal_oversimplification', 
    'doubt', 'loaded_language', 'name_calling,labeling', 
    'repetition', 'exaggeration,minimisation'
]

# Variation 1 Tagset (3-Class Boundary Detection)
BIO_3_CLASS = ['O', 'B-Propaganda', 'I-Propaganda']
V1_TAG_TO_IDX = {tag: i for i, tag in enumerate(BIO_3_CLASS)}
V1_IDX_TO_TAG = {i: tag for tag, i in V1_TAG_TO_IDX.items()}

# Variation 2 Tagset (17-Class Joint Detection)
BIO_17_CLASS = ['O'] + [f"B-{t}" for t in TECHNIQUES] + [f"I-{t}" for t in TECHNIQUES]
V2_TAG_TO_IDX = {tag: i for i, tag in enumerate(BIO_17_CLASS)}
V2_IDX_TO_TAG = {i: tag for tag, i in V2_TAG_TO_IDX.items()}

# Classifier Head Tagset
TECH_TO_IDX = {tech: i for i, tech in enumerate(TECHNIQUES)}
IDX_TO_TECH = {i: tech for tech, i in TECH_TO_IDX.items()}

In [ ]:
# ==========================================
# CELL 3: Task 2 Data Pipeline
# ==========================================

class Task2DataPipeline:
    """
    Processes raw TSV rows into DeBERTa subword encodings, aligned BIO tags,
    and topological features for the baseline model.
    Handles <BOS> and <EOS> markers present across all rows (including not_propaganda).
    """
    def __init__(self, model_checkpoint="microsoft/deberta-v3-xsmall"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        
    def universal_cleaning_t2(self, raw_text: str) -> str:
        """Cleans text while preserving case and punctuation for Transformer context."""
        text = raw_text.strip()
        text = text.replace("\\'", "'").replace('\\"', '"') # Escape characters
        text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
        text = re.sub(r'[\\/\[\]*|@$#+=]', ' ', text) # Strip digital artifacts
        return " ".join(text.split())

    def parse_span_and_clean(self, raw_text: str, label: str):
        """
        Extracts exact character offsets of the snippet bounded by <BOS>/<EOS>.
        Every row in the dataset contains <BOS>/<EOS> tags.
        is_active_propaganda is True ONLY if the row's label is NOT 'not_propaganda'.
        """
        text = self.universal_cleaning_t2(raw_text)
        
        bos_idx = text.find("<BOS>")
        eos_idx = text.find("<EOS>")
        
        if bos_idx != -1 and eos_idx != -1:
            pre_bos = text[:bos_idx]
            inside = text[bos_idx+5:eos_idx]    # +5 to skip over '<BOS>' string length
            post_eos = text[eos_idx+5:]         # +5 to skip over '<EOS>' string length
            
            clean_text = pre_bos + inside + post_eos
            char_start = len(pre_bos)
            char_end = len(pre_bos) + len(inside)
            
            is_active_propaganda = 1.0 if label != 'not_propaganda' else 0.0
        else:
            print("ERROR: Instance has no tags")
            clean_text = text.replace("<BOS>", "").replace("<EOS>", "")
            char_start, char_end = -1, -1
            is_active_propaganda = 0.0
            
        return clean_text, char_start, char_end, is_active_propaganda

    def get_topological_features(self, clean_text: str, char_start: int, char_end: int):
        """
        Extracts language-blind structural metrics for the baseline features.
        Accepts output from parse_span_and_clean()
        """
        tokens = clean_text.split()
        L_tokens = len(tokens) if len(tokens) > 0 else 1
        L_chars = len(clean_text) if len(clean_text) > 0 else 1
        
        word_lengths = [len(w) for w in tokens]
        mu_len = np.mean(word_lengths) if word_lengths else 0.0
        sigma2_len = np.var(word_lengths) if word_lengths else 0.0
        
        cap_ratio = sum(1 for c in clean_text if c.isupper()) / L_chars
        punc_density = sum(1 for c in clean_text if c in string.punctuation) / L_tokens
        digit_ratio = sum(1 for c in clean_text if c.isdigit()) / L_tokens
        
        # Relative bounding ratios (0.0 to 1.0)
        r_start = char_start / L_chars if char_start != -1 else 0.0
        r_end = char_end / L_chars if char_end != -1 else 0.0
        
        x_topo = [L_tokens, L_chars, mu_len, sigma2_len, cap_ratio, punc_density, digit_ratio]
        return torch.tensor(x_topo, dtype=torch.float32), r_start, r_end

    def align_bio_tags(self, clean_text: str, char_start: int, char_end: int, label: str, mode: str):
        """
        Tokenizes clean_text and aligns subword offset mappings to character spans.
        self.tokenizer = AutoTokenizer
        If label == 'not_propaganda', all tokens are assigned the 'O' tag (ID 0).
        """
        encoding = self.tokenizer(
            clean_text, 
            return_offsets_mapping=True, 
            truncation=True, 
            max_length=512,
            return_tensors="pt"
        )
        offsets = encoding['offset_mapping'][0].tolist() # original character indices: [(start,end)]
        
        tag_ids = []
        is_in_span = False
        
        for idx, (o_start, o_end) in enumerate(offsets):
            if o_start == o_end: # zero-length tokens or special boundary: [CLS], [SEP], Padding
                tag_ids.append(0)
                continue
                
            # Assign B-/I- tags ONLY if label is an active propaganda technique
            if (label != 'not_propaganda' and 
                char_start != -1 and    # real token only
                char_start <= o_start   # start of token is within span start
                and o_end <= char_end   # end of token is within span end
                ):
                if not is_in_span:  # update pre-init loop holder
                    tag = "B-Propaganda" if mode == "var1" else f"B-{label}"
                    is_in_span = True
                else:
                    tag = "I-Propaganda" if mode == "var1" else f"I-{label}"
                    is_in_span = True
                
                tag_map = V1_TAG_TO_IDX if mode == "var1" else V2_TAG_TO_IDX
                tag_ids.append(tag_map.get(tag, 0))     # convert tags to numerical ID
            else:
                tag_ids.append(0) # 'O' tag for non-propaganda text or outside span
                is_in_span = False

        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        tags_tensor = torch.tensor([tag_ids], dtype=torch.long)     # conver to pytorch format
        
        # assertion check: Ensure input_ids, attention_mask, and tag_ids match in sequence length
        assert input_ids.shape[1] == attention_mask.shape[1] == tags_tensor.shape[1], (
            f"Dimension mismatch! input_ids: {input_ids.shape[1]}, "
            f"attention_mask: {attention_mask.shape[1]}, tags: {tags_tensor.shape[1]}"
        )
                
        return input_ids, attention_mask, tags_tensor

In [ ]:
# ==========================================
# CELL 4: Cascading Window Qualification Router (Evaluation)
# ==========================================

class Task2Evaluator:
    
    @staticmethod
    def get_tolerance(span_length: int) -> int:
        if span_length <= 5: return 0
        elif span_length <= 10: return 1
        elif span_length <= 15: return 2
        elif span_length <= 50: return 2 + ((span_length - 15) // 5)
        else: return 10

    @staticmethod
    def evaluate_predictions(gold_data: list, pred_data: list):
        """
        gold_data/pred_data format: [{"span": (start_idx, end_idx), "technique": "doubt"}, ...]
        Returns Macro-F1 and False Positive / False Negative diagnostic logs.
        """
        y_true = []
        y_pred = []
        error_logs = [] # verbose capture of errors
        
        for gold, pred in zip(gold_data, pred_data):
            
            # If no propaganda exists in gold and none predicted
            if gold["technique"] == "not_propaganda" and pred["span"] == (-1, -1):
                continue # True Negative, ignored in Macro-F1
                
            # False Positive: Hallucinated span on neutral text
            if gold["technique"] == "not_propaganda" and pred["span"] != (-1, -1):
                y_true.append("not_propaganda")
                y_pred.append(pred["technique"])
                error_logs.append({"error": "Hallucinated Span", "pred": pred})     # logging the error pred class
                continue
                
            # False Negative: Missed span entirely
            if gold["technique"] != "not_propaganda" and pred["span"] == (-1, -1):
                y_true.append(gold["technique"])
                y_pred.append("not_propaganda")
                error_logs.append({"error": "Missed Span", "gold": gold})
                continue

            # Both exist: Apply Cascading Window Router
            g_start, g_end = gold["span"]
            p_start, p_end = pred["span"]
            gold_len = g_end - g_start + 1
            delta = Task2Evaluator.get_tolerance(gold_len)
            
            if abs(p_start - g_start) <= delta and abs(p_end - g_end) <= delta: # tolerance at both ends
                # Boundary Qualified
                y_true.append(gold["technique"])
                y_pred.append(pred["technique"])
                if gold["technique"] != pred["technique"]:
                    error_logs.append({"error": "Technique Misclassification", "gold": gold, "pred": pred})
            else:
                # Boundary Disqualified
                left_failed = abs(p_start - g_start) > delta
                right_failed = abs(p_end - g_end) > delta
                
                if left_failed and right_failed:
                    failure_subtype = "Boundary Failure: Both Left and Right"
                elif left_failed:
                    failure_subtype = "Boundary Failure: Left Only (Start Offset)"
                else:
                    failure_subtype = "Boundary Failure: Right Only (End Offset)"

                # Macro F-1 (TODO: Explain this in draft)
                y_true.append(gold["technique"])
                y_pred.append("not_propaganda")     # FN for gold, recall penalise
                
                y_true.append("not_propaganda")     # synthetic evaluation record
                y_pred.append(pred["technique"])    # FP for pred
                # the model predicts a span with didnt qualify as a span and therefore
                # predicted something pred["technique"] on a "neutral" span
                # TODO: this may be a limitation if the boundary settings are not correct
                # misclassification: correct token boundaries, wrong label.
                # boundary failure: span localization missed entirely
                
                error_logs.append({
                    "error": "Boundary Localization Failure", 
                    "subtype": failure_subtype,
                    "gold": gold, 
                    "pred": pred,
                    "delta_allowed": delta,
                    "left_diff": abs(p_start - g_start),
                    "right_diff": abs(p_end - g_end)
                })

        # Calculate Macro-F1 (ignoring 'not_propaganda' as a target class)
        valid_labels = TECHNIQUES
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=valid_labels, average='macro', zero_division=0
        )
        
        return {"Macro-F1": f1, "Precision": precision, "Recall": recall, "Error_Logs": error_logs}
    
    # TODO: where is the class level analysis

In [ ]:
# ==========================================
# CELL 5: Topological Baseline & DeBERTa Models
# ==========================================

# --- TASK 2 BASELINE ---
class TopologicalBaseline(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(7, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 3) # Outputs: [P_prop_logit, R_start, R_end]
        )
        
    def forward(self, x_topo):
        return self.mlp(x_topo)

# --- DEBERTA-CRF TAGGER (Used for Var 1 & Var 2) ---
class DebertaCRFTagger(nn.Module):
    def __init__(self, num_tags, model_checkpoint="microsoft/deberta-v3-xsmall"):
        super().__init__()
        self.deberta = DebertaV2Model.from_pretrained(model_checkpoint)
        self.hidden2tag = nn.Linear(self.deberta.config.hidden_size, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        
    def forward(self, input_ids, attention_mask, tags=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        emissions = self.hidden2tag(sequence_output)
        
        if tags is not None:
            # Negative Log-Likelihood Loss for Training
            loss = -self.crf(emissions, tags, mask=attention_mask.byte(), reduction='mean')
            return loss
        else:
            # Viterbi Decoding for Inference
            return self.crf.decode(emissions, mask=attention_mask.byte())

# --- VARIATION 1 STAGE 2: SPAN CLASSIFIER ---
class SpanClassifierHead(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=64, num_classes=8): # 384 for deberta-v3-xsmall
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, span_embedding):
        return self.mlp(span_embedding)

In [ ]:
# ==========================================
# CELL 6: Execution Engine (Updated calls)
# ==========================================
class Task2Executor:
    def __init__(self, pipeline: Task2DataPipeline):
        self.pipeline = pipeline
        
    def extract_viterbi_span(self, viterbi_path, mode="var2"):
        """Extracts the first continuous non-O span and its technique from the sequence."""
        start_idx, end_idx, technique = -1, -1, "not_propaganda"
        
        idx_to_tag = V1_IDX_TO_TAG if mode == "var1" else V2_IDX_TO_TAG
        
        for i, tag_id in enumerate(viterbi_path):
            tag = idx_to_tag[tag_id]
            if tag.startswith("B-"):
                start_idx = i
                end_idx = i
                technique = "Propaganda" if mode == "var1" else tag.split("-")[1]
            elif tag.startswith("I-") and start_idx != -1:
                end_idx = i
                
        return start_idx, end_idx, technique

    def train_baseline(self, train_path, epochs=5):
        print("\n--- Training Topological Baseline ---")
        model = TopologicalBaseline()
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
        bce_loss = nn.BCEWithLogitsLoss()
        mse_loss = nn.MSELoss()
        
        model.train()
        for epoch in range(epochs):
            total_loss = 0.0
            with open(train_path, mode='r', encoding='utf-8') as file:
                reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
                for row in reader:
                    label = row['label']
                    clean_text, c_start, c_end, is_prop = self.pipeline.parse_span_and_clean(row['tagged_in_context'], label)
                    x_topo, r_start, r_end = self.pipeline.get_topological_features(clean_text, c_start, c_end)
                    
                    optimizer.zero_grad()
                    out = model(x_topo.unsqueeze(0))
                    
                    p_logit = out[0, 0]
                    r_preds = out[0, 1:]
                    r_targs = torch.tensor([r_start, r_end], dtype=torch.float32)
                    
                    loss = bce_loss(p_logit, torch.tensor(is_prop, dtype=torch.float32))
                    if is_prop == 1.0: # Penalize boundary regression loss on active propaganda
                        loss += mse_loss(r_preds, r_targs)
                        
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
            print(f"Epoch {epoch+1} | Total Baseline Loss: {total_loss:.4f}")
        return model

    def train_deberta_tagger(self, train_path, mode="var2", epochs=3):
        print(f"\n--- Training DeBERTa-CRF ({mode.upper()}) ---")
        num_tags = 3 if mode == "var1" else 17
        model = DebertaCRFTagger(num_tags=num_tags)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
        
        model.train()
        for epoch in range(epochs):
            total_loss = 0.0
            with open(train_path, mode='r', encoding='utf-8') as file:
                reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
                for i, row in enumerate(reader, start=1):
                    label = row['label']
                    clean_text, c_start, c_end, _ = self.pipeline.parse_span_and_clean(row['tagged_in_context'], label)
                    input_ids, att_mask, tags = self.pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode)
                    
                    if input_ids.shape[1] > 256: continue # Safety length cap
                    
                    optimizer.zero_grad()
                    loss = model(input_ids, att_mask, tags)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                    
                    if i % 100 == 0:
                        print(f"  Processed {i} instances...")
                        
            print(f"Epoch {epoch+1} | Total CRF NLL Loss: {total_loss:.4f}")
        return model
        
    def evaluate_tagger(self, model, test_path, mode="var2"):
        model.eval()
        gold_data, pred_data = [], []
        
        with open(test_path, mode='r', encoding='utf-8') as file:
            reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in reader:
                label = row['label']
                clean_text, c_start, c_end, is_prop = self.pipeline.parse_span_and_clean(row['tagged_in_context'], label)
                input_ids, att_mask, tags = self.pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode)
                
                # Gold span & label extraction
                g_start, g_end, _ = self.extract_viterbi_span(tags[0].tolist(), mode)
                gold_data.append({"span": (g_start, g_end), "technique": label})
                
                # Model Viterbi Path Prediction
                with torch.no_grad():
                    viterbi_path = model(input_ids, att_mask)[0]
                
                p_start, p_end, p_tech = self.extract_viterbi_span(viterbi_path, mode)
                pred_data.append({"span": (p_start, p_end), "technique": p_tech})
                
        eval_results = Task2Evaluator.evaluate_predictions(gold_data, pred_data)
        print(f"\n==========================================")
        print(f" TASK 2 EVALUATION REPORT ({mode.upper()})")
        print(f"==========================================")
        print(f" Macro-F1 Score: {eval_results['Macro-F1']:.4f}")
        print(f" Macro Precision: {eval_results['Precision']:.4f}")
        print(f" Macro Recall:    {eval_results['Recall']:.4f}")
        print(f" Total Logged Errors: {len(eval_results['Error_Logs'])}")
        print(f"==========================================\n")
        return eval_results

In [ ]:
# ==========================================
# CELL 7: Running Task 2
# ==========================================

# Initialize
t2_pipeline = Task2DataPipeline()
executor = Task2Executor(t2_pipeline)

# 1. Topological Baseline
baseline_model = executor.train_baseline('../data/propaganda_train_100.tsv', epochs=5)

# 2. Variation 2 (17-Class Joint Tagger)
# Using a small subset file here for rapid testing. Switch to full train.tsv for full run.
var2_model = executor.train_deberta_tagger('../data/propaganda_train_100.tsv', mode="var2", epochs=3)

# Evaluate Variation 2
var2_results = executor.evaluate_tagger(var2_model, '../data/propaganda_val.tsv', mode="var2")

# To implement Variation 1 Stage 2, you would extract hidden_states from var1_model 
# and train SpanClassifierHead on the mean-pooled vectors.